# 🚀 QLoRA: ajusta un modelo de lenguaje en la GPU gratis
*Curso de IA Generativa · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

En el tema «🛠 LoRA real» ajustaste **SmolLM2-135M en tu CPU** para que respondiera como el asistente de la tienda
Lumen. Aquí haces lo mismo con un modelo **11 veces más grande**, Qwen2.5-1.5B-Instruct (Apache 2.0), con **QLoRA**:
el modelo base se carga en **4 bits** y solo se entrenan unos pequeños adaptadores **LoRA**.

**Datos:** los mismos 40 ejemplos de la tienda Lumen del curso (34 para entrenar y 6 de productos que el modelo no
ve), para que compares con el resultado en CPU.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os, gc, time
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "|", torch.cuda.get_device_name(0) if DISPOSITIVO == "cuda" else "")
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

def liberar():
    """Devuelve la memoria de la GPU después de borrar un modelo («del modelo»)."""
    gc.collect()
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()

In [ ]:
%pip install -q transformers peft bitsandbytes accelerate

In [ ]:
from pathlib import Path
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/MiAtlas-GenAI")
else:
    CARPETA = Path("MiAtlas-GenAI")
CARPETA.mkdir(parents=True, exist_ok=True)

In [ ]:
import json, re, urllib.request
from pathlib import Path

local = Path("finetune_tienda.jsonl")
if local.exists():
    lineas = local.read_text(encoding="utf-8").splitlines()
else:
    url = "https://raw.githubusercontent.com/psalgadm7/miatlas-cuadernos/main/datos/finetune_tienda.jsonl"
    lineas = urllib.request.urlopen(url).read().decode("utf-8").splitlines()
ejemplos = [json.loads(l)["messages"] for l in lineas if l.strip()]
entrenamiento, prueba = ejemplos[:34], ejemplos[34:]
print(len(entrenamiento), "para entrenar ·", len(prueba), "de prueba")
print(ejemplos[0])

## 1. El modelo base en 4 bits

Sin GPU, el cuaderno usa SmolLM2-135M (el del tema en CPU) para que igual puedas seguir los pasos.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if DISPOSITIVO == "cuda":
    MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
    cuantizacion = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                      bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    modelo = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=cuantizacion, device_map="auto")
else:
    MODELO = "HuggingFaceTB/SmolLM2-135M-Instruct"
    modelo = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float32)
tokenizador = AutoTokenizer.from_pretrained(MODELO)
print(f"{MODELO} · {sum(p.numel() for p in modelo.parameters()) / 1e6:.0f} M parámetros")
if DISPOSITIVO == "cuda":
    print(f"Memoria en la GPU: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
def responder(m, mensajes, tokens=60):
    entrada = tokenizador.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to(m.device)
    with torch.no_grad():
        salida = m.generate(entrada, max_new_tokens=tokens, do_sample=False, pad_token_id=tokenizador.eos_token_id)
    return tokenizador.decode(salida[0, entrada.shape[1]:], skip_special_tokens=True).strip()

def precio(texto):
    m = re.search(r"\$[\d.]+", texto)
    return m[0].rstrip(".") if m else None

def evaluar(m, titulo):
    print(f"── {titulo}")
    formato = precios = 0
    for mensajes in prueba:
        esperado, obtenido = mensajes[-1]["content"], responder(m, mensajes[:-1])
        formato += obtenido.endswith("— Equipo Lumen")
        precios += precio(obtenido) == precio(esperado)
        print(f"   ❓ {mensajes[1]['content']}\n   🤖 {obtenido[:110]!r}")
    print(f"   formato de la tienda: {formato}/{len(prueba)} · precio correcto: {precios}/{len(prueba)}\n")
    return formato, precios

antes = evaluar(modelo, "ANTES: el modelo base")

## 2. Preparar los datos: aprender solo de la respuesta

Igual que en el tema en CPU: el texto completo entra al modelo, pero la pérdida se calcula **solo sobre la respuesta**
(las etiquetas del resto valen -100).

In [ ]:
def preparar(mensajes):
    prompt = tokenizador.apply_chat_template(mensajes[:-1], add_generation_prompt=True)
    completo = tokenizador.apply_chat_template(mensajes)
    etiquetas = [-100] * len(prompt) + completo[len(prompt):]
    return torch.tensor([completo]), torch.tensor([etiquetas])

lotes = [preparar(m) for m in entrenamiento]
print("Un ejemplo:", lotes[0][0].shape[1], "tokens; se aprende de", (lotes[0][1] != -100).sum().item())

## 3. Los adaptadores LoRA

En vez de cambiar los miles de millones de pesos, LoRA agrega a algunas capas dos matrices pequeñas (de rango `r`) y
entrena solo esas. Con QLoRA, el modelo base además queda congelado en 4 bits.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if DISPOSITIVO == "cuda":
    modelo = prepare_model_for_kbit_training(modelo)
config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
modelo = get_peft_model(modelo, config)
modelo.print_trainable_parameters()

## 4. Entrenar

In [ ]:
EPOCAS = 1 if PRUEBA_RAPIDA else 3
optimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=2e-4)
modelo.train()
inicio = time.perf_counter()
for epoca in range(EPOCAS):
    perdidas = []
    for i in torch.randperm(len(lotes)).tolist():
        ids, etiquetas = (t.to(modelo.device) for t in lotes[i])
        perdida = modelo(input_ids=ids, labels=etiquetas).loss
        perdida.backward(); optimizador.step(); optimizador.zero_grad()
        perdidas.append(perdida.item())
    print(f"Época {epoca + 1}/{EPOCAS} · pérdida media {sum(perdidas) / len(perdidas):.3f} · {time.perf_counter() - inicio:.0f} s")
modelo.eval()
despues = evaluar(modelo, "DESPUÉS: con los adaptadores LoRA")
print(f"Formato: {antes[0]} → {despues[0]} de {len(prueba)} · precio: {antes[1]} → {despues[1]} de {len(prueba)}")

> **Compara con el tema en CPU:** ¿acierta más precios un modelo más grande? El formato se aprende rápido; los datos
> nuevos (precios de productos que no vio) no se pueden «aprender»: para eso está RAG.

## 5. Guardar los adaptadores en Drive (y, si quieres, publicarlos)

In [ ]:
destino = CARPETA / "adaptadores-lumen"
modelo.save_pretrained(destino)               # solo los adaptadores: unos MB, no el modelo entero
tokenizador.save_pretrained(destino)
print("Guardados en", destino, f"({sum(f.stat().st_size for f in destino.rglob('*') if f.is_file()) / 1024**2:.1f} MB)")

# Opcional: subirlos a TU cuenta de Hugging Face (necesita HF_TOKEN con permiso de escritura en Secretos)
# modelo.push_to_hub("tu-usuario/qwen2.5-1.5b-lumen-lora", private=True)

### Para usarlos después

Carga el modelo base (en 16 bits si vas a unirlos) y aplica los adaptadores con
`PeftModel.from_pretrained(base, destino)`. `merge_and_unload()` los funde en un modelo normal, más rápido para
responder.

### Con más tiempo de GPU

Para un ajuste más general en español, prueba con `bertin-project/alpaca-spanish` (52.000 instrucciones, CC BY 4.0):
toma unos miles, conviértelos al formato `messages` y repite los pasos 2 a 4.

## Pruébalo tú

Repite el ajuste con `r=4` y con `r=64`. Guarda en `parametros_r4` cuántos parámetros entrena LoRA con `r=4`. ¿Cambia
el resultado en la tienda?

In [ ]:
parametros_r4 = None   # ← tu código (pista: modelo.get_nb_trainable_parameters())

In [ ]:
# Comprobación
if parametros_r4 is None:
    print("Aún no lo resuelves.")
elif parametros_r4 > 0:
    print("✅ Registrado: compáralo con r=16")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- **QLoRA** = modelo base en 4 bits congelado + adaptadores LoRA que sí se entrenan.
- Se guardan solo los adaptadores (MB), no el modelo (GB).
- El ajuste fino enseña **forma y estilo**; para datos que cambian, mejor RAG.

**Fuentes:** [PEFT · LoRA](https://huggingface.co/docs/peft/developer_guides/lora) · [PEFT · cuantización (QLoRA)](https://huggingface.co/docs/peft/developer_guides/quantization) · [Dettmers et al. (2023), QLoRA](https://arxiv.org/abs/2305.14314) · [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) · [alpaca-spanish (CC BY 4.0)](https://huggingface.co/datasets/bertin-project/alpaca-spanish)